# Put CloudFront in front of an S3 origin, configure cache behaviors, and prove cache hits cut response time

A mobile app team has a public S3 bucket of product images that loads in 200 ms from Seattle and 2 seconds from Sydney. The CTO wants Sydney parity before next quarter's launch in APAC. You have one session to drop CloudFront in front of the existing bucket, lock the bucket policy down so the only path to the object is through the distribution, prove a cold request and a warm request behave the way the AWS docs describe, and hand the team a cache-hit ratio they can use as the production smoke test.

You will build:

- A private S3 bucket (Block Public Access fully on) with one seed object.
- A CloudFront Origin Access Control (OAC) wired into a CloudFront distribution.
- A bucket policy granting `Principal.Service=cloudfront.amazonaws.com` `s3:GetObject` scoped by `Condition.StringEquals.AWS:SourceArn=<distribution ARN>`.
- A CloudFront distribution that reaches `Status=Deployed`.

Then you make two HTTP GETs against the distribution URL and see `X-Cache: Miss from cloudfront` on the first and `X-Cache: Hit from cloudfront` on the second.

**Time.** Plan for about 55 minutes hands-on but expect 30-45 minutes of wall-clock waiting on CloudFront. The distribution takes 5-15 minutes to deploy after you create it, and another 10-15 minutes to disable before you can delete it. Plan for an hour total.

**Cost.** This lab costs effectively zero. CloudFront's free tier swallows the two requests we make, and the seed object is a few hundred bytes in S3. The cost that matters here is not money; it is time. Plan for the deploy + disable-and-delete cycle.

This lab maps to AWS SAA-C03 Domain 3 (Design High-Performing Architectures, 24%).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned per LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import datetime as dt
import getpass
import json
import time
from urllib.error import URLError
from urllib.request import Request, urlopen

import boto3
from botocore.exceptions import ClientError
from IPython.display import clear_output

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "cloudfront-origin-and-cache-behaviors"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"
SEED_KEY = "seed.jpg"
OAC_NAME = f"labrun-{LAB_ID}-oac"

BUCKET_NAME = None
LAB_STATE = {
    "session_start": None,
    "oac_id": None,
    "distribution_id": None,
    "distribution_arn": None,
    "distribution_domain": None,
}

In [ ]:
# NBVAL_SKIP
# Register the session and validate AWS credentials.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")
print(f"Bucket name: {BUCKET_NAME}")

LAB_STATE["session_start"] = dt.datetime.now(dt.timezone.utc)
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan.
# CloudFront distribution carries critical=True for the duration of the lab
# because disable-then-delete is the slow path; the cloudfront_distribution
# adapter handles the 10-20 minute disable poll + ETag refresh + delete.

CLEANUP_MANIFEST = []


def _rebuild_manifest():
    CLEANUP_MANIFEST.clear()
    if BUCKET_NAME:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="s3_bucket_policy",
                id=BUCKET_NAME,
                region=REGION,
                cli_delete_command=(
                    f"aws s3api delete-bucket-policy --bucket {BUCKET_NAME}"
                ),
            )
        )
    if LAB_STATE["distribution_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="cloudfront_distribution",
                id=LAB_STATE["distribution_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws cloudfront get-distribution-config --id "
                    f"{LAB_STATE['distribution_id']} --query ETag --output text "
                    f"| xargs -I {{}} aws cloudfront delete-distribution "
                    f"--id {LAB_STATE['distribution_id']} --if-match {{}}"
                ),
            )
        )
    if LAB_STATE["oac_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="cloudfront_origin_access_control",
                id=LAB_STATE["oac_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws cloudfront delete-origin-access-control --id "
                    f"{LAB_STATE['oac_id']} "
                    f"--if-match $(aws cloudfront get-origin-access-control --id "
                    f"{LAB_STATE['oac_id']} --query ETag --output text)"
                ),
            )
        )
    if BUCKET_NAME:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="s3_object",
                id=SEED_KEY,
                parent=BUCKET_NAME,
                region=REGION,
                cli_delete_command=(
                    f"aws s3api delete-object --bucket {BUCKET_NAME} --key {SEED_KEY}"
                ),
            )
        )
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="s3_bucket",
                id=BUCKET_NAME,
                region=REGION,
                cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
            )
        )


_rebuild_manifest()


def _atexit_cleanup():
    try:
        _rebuild_manifest()
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to provision.")

## Task 1: Create the private S3 bucket and seed object

Block Public Access fully on at creation. Tag for the orphan scan. Upload a tiny seed object that stands in for the product image: any bytes are fine, the lab does not inspect content.

The observe cell confirms the bucket, the public-access-block configuration, and the seed object are all in place. No long poll; the bucket is available immediately.

In [ ]:
# NBVAL_SKIP
# Task 1: create the bucket with Block Public Access, tag, and upload the seed object.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket.
# YOUR CODE: call s3.create_bucket(Bucket=BUCKET_NAME). In us-east-1, do not
# pass CreateBucketConfiguration. Handle BucketAlreadyOwnedByYou as a no-op.

# Tag the bucket with the lab slug.
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

# Block Public Access (all four flags True).
# YOUR CODE: call s3.put_public_access_block(Bucket=BUCKET_NAME,
# PublicAccessBlockConfiguration={"BlockPublicAcls": True, "IgnorePublicAcls":
# True, "BlockPublicPolicy": True, "RestrictPublicBuckets": True}).

# Upload the seed object.
# YOUR CODE: call s3.put_object(Bucket=BUCKET_NAME, Key=SEED_KEY,
# Body=b"labrun cloudfront origin seed object\n", ContentType="image/jpeg").

_rebuild_manifest()
print(f"Bucket created: s3://{BUCKET_NAME}")
print(f"Public access:  blocked (all four flags True)")
print(f"Seed object:    s3://{BUCKET_NAME}/{SEED_KEY}")

In [ ]:
# NBVAL_SKIP
# Observe: snapshot the bucket public-access-block + tagging + seed object.

s3_obs = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

clear_output(wait=True)
try:
    pub = s3_obs.get_public_access_block(Bucket=BUCKET_NAME)["PublicAccessBlockConfiguration"]
except ClientError as e:
    pub = {"error": str(e)}
try:
    tags = s3_obs.get_bucket_tagging(Bucket=BUCKET_NAME)["TagSet"]
except ClientError:
    tags = []
try:
    head = s3_obs.head_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
    seed_size = head.get("ContentLength")
except ClientError as e:
    seed_size = f"missing ({e})"

print(f"Bucket: s3://{BUCKET_NAME}")
print("-" * 64)
print(f"  Public access block:")
for k in ("BlockPublicAcls", "IgnorePublicAcls", "BlockPublicPolicy", "RestrictPublicBuckets"):
    print(f"    {k:24}  {pub.get(k)}")
print(f"  Tags: {tags}")
print(f"  Seed object size: {seed_size}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: bucket exists, all four public-access flags True, seed object present.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            tags = {t["Key"]: t["Value"] for t in s3c.get_bucket_tagging(Bucket=BUCKET_NAME).get("TagSet", [])}
        except ClientError as e:
            if e.response.get("Error", {}).get("Code") == "NoSuchTagSet":
                tags = {}
            else:
                return CheckpointResult(status="error", error_reason=str(e))
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bucket missing {LAB_TAG_KEY}={LAB_TAG_VALUE}. Found {tags}."
                ),
            )
        pub = s3c.get_public_access_block(Bucket=BUCKET_NAME)["PublicAccessBlockConfiguration"]
        flags = ("BlockPublicAcls", "IgnorePublicAcls", "BlockPublicPolicy", "RestrictPublicBuckets")
        missing = [k for k in flags if not pub.get(k)]
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Public access block flags not all True. Missing: {missing}."
                ),
            )
        try:
            s3c.head_object(Bucket=BUCKET_NAME, Key=SEED_KEY)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Seed object {SEED_KEY!r} missing from bucket: {e}",
            )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Three S3 calls. `create_bucket`, `put_public_access_block`, `put_object`. The tag is already wired in this cell's prelude.

</details>

<details><summary>Hint 2 (direction)</summary>

`put_public_access_block` takes a `PublicAccessBlockConfiguration` dict with all four boolean flags. All four must be `True`; any one set to `False` leaves a way for an ACL or a stray policy to expose the bucket.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "BucketAlreadyOwnedByYou":
        raise

s3.put_public_access_block(
    Bucket=BUCKET_NAME,
    PublicAccessBlockConfiguration={
        "BlockPublicAcls": True,
        "IgnorePublicAcls": True,
        "BlockPublicPolicy": True,
        "RestrictPublicBuckets": True,
    },
)

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=SEED_KEY,
    Body=b"labrun cloudfront origin seed object\n",
    ContentType="image/jpeg",
)
```

</details>

**Wallet check.** $0.00. The bucket, public-access-block, and tiny seed object all fit under the always-free S3 tier.

## Task 2: Create the OAC and the CloudFront distribution

Two CloudFront calls:

1. `cloudfront.create_origin_access_control(...)` with `SigningProtocol=sigv4` and `OriginAccessControlOriginType=s3`.
2. `cloudfront.create_distribution_with_tags(...)` with one origin pointed at the S3 bucket's regional endpoint and the OAC ID. Default cache behavior, no signed URLs, no geo restrictions.

The distribution carries `S3OriginConfig.OriginAccessIdentity=""` (empty string is the AWS-required signal that the old OAI mechanism is NOT in use; OAC supersedes OAI). The origin's `OriginAccessControlId` carries the OAC ID.

The observe cell polls until `Status=Deployed`. Typical wait 5-15 minutes.

In [ ]:
# NBVAL_SKIP
# Task 2: OAC + CloudFront distribution.

cf = boto3.client(
    "cloudfront",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the OAC.
# YOUR CODE: call cf.create_origin_access_control(OriginAccessControlConfig={
#     "Name": OAC_NAME,
#     "Description": "labrun lab 9 OAC for S3",
#     "SigningProtocol": "sigv4",
#     "SigningBehavior": "always",
#     "OriginAccessControlOriginType": "s3",
# }). Capture OriginAccessControl.Id in LAB_STATE["oac_id"]. Handle the
# OriginAccessControlAlreadyExists case by listing OACs and finding the
# matching Name.

# Create the distribution.
caller_reference = f"labrun-{LAB_ID}-{int(time.time())}"
bucket_regional_domain = f"{BUCKET_NAME}.s3.{REGION}.amazonaws.com"
distribution_config = {
    "CallerReference": caller_reference,
    "Comment": "labrun lab 9 CloudFront + S3 + OAC",
    "Enabled": True,
    "Origins": {
        "Quantity": 1,
        "Items": [
            {
                "Id": "labrun-s3-origin",
                "DomainName": bucket_regional_domain,
                "OriginPath": "",
                "CustomHeaders": {"Quantity": 0},
                "S3OriginConfig": {"OriginAccessIdentity": ""},
                "OriginAccessControlId": LAB_STATE["oac_id"] or "",
                "ConnectionAttempts": 3,
                "ConnectionTimeout": 10,
                "OriginShield": {"Enabled": False},
            }
        ],
    },
    "DefaultCacheBehavior": {
        "TargetOriginId": "labrun-s3-origin",
        "ViewerProtocolPolicy": "allow-all",
        "AllowedMethods": {
            "Quantity": 2,
            "Items": ["GET", "HEAD"],
            "CachedMethods": {"Quantity": 2, "Items": ["GET", "HEAD"]},
        },
        "CachePolicyId": "658327ea-f89d-4fab-a63d-7e88639e58f6",  # CachingOptimized managed policy
        "Compress": True,
        "SmoothStreaming": False,
        "FieldLevelEncryptionId": "",
    },
    "PriceClass": "PriceClass_100",
    "ViewerCertificate": {"CloudFrontDefaultCertificate": True},
    "Restrictions": {"GeoRestriction": {"RestrictionType": "none", "Quantity": 0}},
    "WebACLId": "",
    "HttpVersion": "http2",
    "IsIPV6Enabled": False,
}
tags = {
    "Items": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
}

# YOUR CODE: call cf.create_distribution_with_tags(DistributionConfigWithTags={
#     "DistributionConfig": distribution_config,
#     "Tags": tags,
# }). Capture Distribution.Id, Distribution.ARN, and Distribution.DomainName
# in LAB_STATE. Handle DistributionAlreadyExists by listing and matching the
# CallerReference. Distribution provisioning is asynchronous; the observe
# cell handles the wait.

_rebuild_manifest()
print(f"OAC:               {LAB_STATE['oac_id']}")
print(f"Distribution ID:   {LAB_STATE['distribution_id']}")
print(f"Distribution ARN:  {LAB_STATE['distribution_arn']}")
print(f"Distribution domain: {LAB_STATE['distribution_domain']}")

In [ ]:
# NBVAL_SKIP
# Observe: poll the distribution until Status=Deployed. Typical 5-15 minutes;
# ceiling 20 minutes.

cf_obs = boto3.client(
    "cloudfront",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 1200  # 20-minute ceiling
status = "?"
while time.time() < deadline:
    clear_output(wait=True)
    try:
        dist = cf_obs.get_distribution(Id=LAB_STATE["distribution_id"])["Distribution"]
        status = dist.get("Status")
        domain = dist.get("DomainName")
        last_modified = dist.get("LastModifiedTime")
    except ClientError as e:
        status = f"error: {e}"
        domain = "?"
        last_modified = "?"
    elapsed = int(1200 - (deadline - time.time()))
    print(f"CloudFront distribution at t+{elapsed:>4}s:")
    print("-" * 64)
    print(f"  ID:            {LAB_STATE['distribution_id']}")
    print(f"  Status:        {status}")
    print(f"  Domain:        {domain}")
    print(f"  LastModified:  {last_modified}")
    if status == "Deployed":
        print()
        print("Distribution is deployed. Move on to Task 3 (bucket policy + cache hit).")
        break
    time.sleep(30)
else:
    print()
    print("Timed out waiting for Status=Deployed. AWS occasionally takes longer than 20")
    print("minutes; try the checkpoint again in another 5-10 minutes.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: OAC exists with the right shape and the distribution
# references it.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        cfc = boto3.client(
            "cloudfront",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        oacs = cfc.list_origin_access_controls()["OriginAccessControlList"]["Items"]
        lab_oac = next(
            (
                o
                for o in oacs
                if o.get("Name") == OAC_NAME
                and o.get("SigningProtocol") == "sigv4"
                and o.get("OriginAccessControlOriginType") == "s3"
            ),
            None,
        )
        if lab_oac is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No OAC named {OAC_NAME!r} with sigv4 + s3 origin type."
                ),
            )

        if not LAB_STATE["distribution_id"]:
            return CheckpointResult(
                status="fail",
                error_reason="Distribution was not created. Re-run the Task 2 cell.",
            )
        config = cfc.get_distribution_config(Id=LAB_STATE["distribution_id"])
        origins = config["DistributionConfig"]["Origins"]["Items"]
        if not origins:
            return CheckpointResult(
                status="fail",
                error_reason="Distribution has no origins.",
            )
        origin = origins[0]
        bucket_regional_domain = f"{BUCKET_NAME}.s3.{REGION}.amazonaws.com"
        if origin.get("DomainName") != bucket_regional_domain:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Origin DomainName is {origin.get('DomainName')!r}, "
                    f"expected {bucket_regional_domain!r}."
                ),
            )
        if origin.get("OriginAccessControlId") != lab_oac["Id"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Distribution origin does not reference the lab OAC. Re-create "
                    "the distribution with OriginAccessControlId=<oac id>."
                ),
            )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two CloudFront calls. `create_origin_access_control` builds the OAC. `create_distribution_with_tags` builds the distribution and tags it in one call (`create_distribution` alone does not accept tags).

</details>

<details><summary>Hint 2 (direction)</summary>

The `OriginAccessControlConfig` needs `SigningProtocol="sigv4"` and `OriginAccessControlOriginType="s3"`. The distribution's `Origins.Items[0]` needs both `S3OriginConfig.OriginAccessIdentity=""` (empty string explicitly opts out of OAI) AND `OriginAccessControlId=<oac id>`. The lab pre-fills most of the DistributionConfig dict; you only need to make the two API calls.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
try:
    resp = cf.create_origin_access_control(
        OriginAccessControlConfig={
            "Name": OAC_NAME,
            "Description": "labrun lab 9 OAC for S3",
            "SigningProtocol": "sigv4",
            "SigningBehavior": "always",
            "OriginAccessControlOriginType": "s3",
        },
    )
    LAB_STATE["oac_id"] = resp["OriginAccessControl"]["Id"]
except ClientError as e:
    if e.response.get("Error", {}).get("Code") == "OriginAccessControlAlreadyExists":
        oacs = cf.list_origin_access_controls()["OriginAccessControlList"]["Items"]
        match = next(o for o in oacs if o.get("Name") == OAC_NAME)
        LAB_STATE["oac_id"] = match["Id"]
    else:
        raise

# Refresh the distribution_config dict with the now-known OAC id.
distribution_config["Origins"]["Items"][0]["OriginAccessControlId"] = LAB_STATE["oac_id"]

resp = cf.create_distribution_with_tags(
    DistributionConfigWithTags={
        "DistributionConfig": distribution_config,
        "Tags": tags,
    },
)
LAB_STATE["distribution_id"] = resp["Distribution"]["Id"]
LAB_STATE["distribution_arn"] = resp["Distribution"]["ARN"]
LAB_STATE["distribution_domain"] = resp["Distribution"]["DomainName"]
```

</details>

**Wallet check.** Still $0.00. CloudFront does not bill at rest; only request volume and data transfer cost money, and the free tier covers the two requests this lab makes.

## Task 3: Attach the OAC bucket policy and wait for the distribution to deploy

The bucket policy is the part that locks the bucket down. The policy:

- Grants `s3:GetObject` on `arn:aws:s3:::<bucket>/*`.
- `Principal.Service=cloudfront.amazonaws.com`.
- `Condition.StringEquals.AWS:SourceArn=<distribution ARN>`. The Condition prevents any CloudFront distribution in any AWS account from reading the bucket; only the distribution whose ARN matches is allowed.

The observe cell from Task 2 already polls until `Status=Deployed`. This task adds the bucket policy and confirms the distribution config matches the deployed state.

In [ ]:
# NBVAL_SKIP
# Task 3: bucket policy granting CloudFront service principal scoped by SourceArn.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowCloudFrontOACAccess",
            "Effect": "Allow",
            "Principal": {"Service": "cloudfront.amazonaws.com"},
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
            "Condition": {
                "StringEquals": {
                    "AWS:SourceArn": LAB_STATE["distribution_arn"],
                }
            },
        }
    ],
}

# YOUR CODE: call s3.put_bucket_policy(Bucket=BUCKET_NAME,
# Policy=json.dumps(bucket_policy)). The Policy parameter must be a JSON string,
# not a dict; json.dumps is required.

print("Bucket policy attached. CloudFront is the only path to the seed object.")
print()
print("If the distribution is not yet Deployed, re-run the observe cell from")
print("Task 2 to watch the status. Once it lands, run Checkpoint 3.")

In [ ]:
# NBVAL_SKIP
# Observe: confirm the bucket policy is in place AND the distribution is
# Deployed. Polls every 30 seconds for up to 5 minutes.

s3_obs = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
cf_obs = boto3.client(
    "cloudfront",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 300
ready = False
while time.time() < deadline:
    clear_output(wait=True)
    try:
        policy = json.loads(s3_obs.get_bucket_policy(Bucket=BUCKET_NAME)["Policy"])
        policy_ok = any(
            s.get("Principal", {}).get("Service") == "cloudfront.amazonaws.com"
            for s in policy.get("Statement", [])
        )
    except ClientError:
        policy = None
        policy_ok = False
    try:
        dist = cf_obs.get_distribution(Id=LAB_STATE["distribution_id"])["Distribution"]
        status = dist.get("Status")
    except ClientError as e:
        status = f"error: {e}"
    elapsed = int(300 - (deadline - time.time()))
    print(f"Origin + distribution at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  Bucket policy present:        {policy is not None}")
    print(f"  Policy allows CloudFront:     {policy_ok}")
    print(f"  Distribution status:          {status}")
    if policy_ok and status == "Deployed":
        ready = True
        print()
        print("Origin is locked to the lab distribution and the distribution is Deployed.")
        break
    time.sleep(30)
else:
    if not ready:
        print()
        print("Either the policy is missing or the distribution is still InProgress.")
        print("Re-run this cell after the distribution reaches Deployed.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: bucket policy allows CloudFront service principal scoped by
# AWS:SourceArn, AND distribution is Deployed.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        cfc = boto3.client(
            "cloudfront",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            policy = json.loads(s3c.get_bucket_policy(Bucket=BUCKET_NAME)["Policy"])
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not read bucket policy: {e}",
            )

        allow_stmt = None
        for s in policy.get("Statement", []):
            principal = s.get("Principal", {})
            actions = s.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]
            if (
                s.get("Effect") == "Allow"
                and principal.get("Service") == "cloudfront.amazonaws.com"
                and any(a == "s3:GetObject" or a == "*" or a == "s3:*" for a in actions)
            ):
                allow_stmt = s
                break
        if allow_stmt is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Bucket policy is missing an Allow statement with "
                    "Principal.Service=cloudfront.amazonaws.com and "
                    "Action=s3:GetObject. Re-run Task 3."
                ),
            )

        source_arn = (allow_stmt.get("Condition") or {}).get("StringEquals", {}).get("AWS:SourceArn")
        if source_arn != LAB_STATE["distribution_arn"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Condition.StringEquals.AWS:SourceArn is missing or wrong. "
                    f"Expected {LAB_STATE['distribution_arn']!r}, got {source_arn!r}. "
                    "Without the SourceArn condition any CloudFront distribution "
                    "could fetch from the bucket."
                ),
            )

        dist = cfc.get_distribution(Id=LAB_STATE["distribution_id"])["Distribution"]
        if dist.get("Status") != "Deployed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Distribution status is {dist.get('Status')!r}, expected "
                    f"Deployed. Re-run the observe cell after the distribution lands."
                ),
            )

        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

One S3 call. The policy is already drafted; you just have to serialize it and attach it.

</details>

<details><summary>Hint 2 (direction)</summary>

`s3.put_bucket_policy(Bucket=BUCKET_NAME, Policy=json.dumps(bucket_policy))`. The Policy parameter is a JSON string, not a dict; `json.dumps` is required. The Condition's `AWS:SourceArn` MUST be the distribution ARN (`LAB_STATE["distribution_arn"]`); without it, any CloudFront distribution in any AWS account could fetch.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
s3.put_bucket_policy(Bucket=BUCKET_NAME, Policy=json.dumps(bucket_policy))
```

</details>

**Wallet check.** Still $0.00. `put_bucket_policy` is a free control-plane call.

## Task 4: Make two GETs against the distribution; second one is X-Cache: Hit

The acid test. The first GET against `https://{distribution_domain}/seed.jpg` is a cache miss; CloudFront fetches from S3, returns the bytes, and caches the response. The second GET (within the cache TTL) returns from the edge cache directly; CloudFront does not touch the origin.

You read the `X-Cache` header on each response. First response: `Miss from cloudfront` or `RefreshHit from cloudfront`. Second response: `Hit from cloudfront`. The body is the same on both requests; only the cache-status indicator changes.

The observe cell loops the second request until `X-Cache: Hit from cloudfront` lands (CloudFront occasionally takes a second request or two to cache for the first time after deploy). Ceiling 2 minutes.

In [ ]:
# NBVAL_SKIP
# Task 4: two HTTPS GETs proving Miss then Hit.

url = f"https://{LAB_STATE['distribution_domain']}/{SEED_KEY}"
print(f"Distribution URL: {url}")
print()


def fetch_x_cache(url):
    """Issue one HTTPS GET and return (status_code, x_cache_header, body_length)."""
    req = Request(url, method="GET")
    with urlopen(req, timeout=10) as resp:
        x_cache = resp.headers.get("X-Cache", "")
        body = resp.read()
    return resp.status, x_cache, len(body)


# YOUR CODE: call fetch_x_cache(url) twice. Sleep 5 seconds between the two
# calls. Print each response's status, X-Cache, and body length. Then assert
# the second X-Cache value contains the substring "Hit from cloudfront".

print("First request (expect Miss from cloudfront):")
first_status, first_xcache, first_size = fetch_x_cache(url)
print(f"  Status: {first_status}")
print(f"  X-Cache: {first_xcache}")
print(f"  Body bytes: {first_size}")

print()
print("Sleeping 5s before the second request...")
time.sleep(5)

print()
print("Second request (expect Hit from cloudfront):")
second_status, second_xcache, second_size = fetch_x_cache(url)
print(f"  Status: {second_status}")
print(f"  X-Cache: {second_xcache}")
print(f"  Body bytes: {second_size}")

In [ ]:
# NBVAL_SKIP
# Observe: loop the second GET until X-Cache contains "Hit from cloudfront",
# capped at 2 minutes. CloudFront occasionally serves the first repeat as
# Miss when the edge hasn't fully populated.

url = f"https://{LAB_STATE['distribution_domain']}/{SEED_KEY}"
deadline = time.time() + 120
hit = False
while time.time() < deadline:
    clear_output(wait=True)
    try:
        with urlopen(Request(url, method="GET"), timeout=10) as resp:
            x_cache = resp.headers.get("X-Cache", "")
            status = resp.status
    except (URLError, Exception) as e:
        x_cache = f"error: {e}"
        status = "?"
    elapsed = int(120 - (deadline - time.time()))
    print(f"CloudFront cache state at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  URL:           {url}")
    print(f"  HTTP status:   {status}")
    print(f"  X-Cache:       {x_cache}")
    if "Hit from cloudfront" in x_cache:
        hit = True
        print()
        print("Cache hit confirmed. The first request seeded the edge, the second one served from cache.")
        break
    time.sleep(10)
else:
    if not hit:
        print()
        print("X-Cache did not flip to Hit within 2 minutes.")
        print("Most likely causes: viewer-protocol-policy mismatch, or the cache")
        print("behavior's TTL is so short the second request expired the entry.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: the distribution URL serves the seed object AND a repeat
# request shows X-Cache: Hit from cloudfront.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        url = f"https://{LAB_STATE['distribution_domain']}/{SEED_KEY}"

        # First GET: warm the cache.
        try:
            with urlopen(Request(url, method="GET"), timeout=15) as resp:
                first_status = resp.status
                first_body_len = len(resp.read())
        except Exception as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"First HTTPS GET against {url} failed: {e}. "
                    f"Distribution may not be Deployed yet."
                ),
            )
        if first_status != 200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"First GET returned HTTP {first_status}, expected 200. The "
                    f"bucket policy or OAC config is wrong."
                ),
            )
        if first_body_len < 1:
            return CheckpointResult(
                status="fail",
                error_reason="First GET returned an empty body; the seed object is missing or empty.",
            )

        # Second GET: poll for X-Cache: Hit.
        deadline = time.time() + 90
        last_xcache = ""
        while time.time() < deadline:
            time.sleep(5)
            try:
                with urlopen(Request(url, method="GET"), timeout=15) as resp:
                    last_xcache = resp.headers.get("X-Cache", "")
            except Exception as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Repeat HTTPS GET failed: {e}",
                )
            if "Hit from cloudfront" in last_xcache:
                return CheckpointResult(status="pass")
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"X-Cache header on repeat requests stayed as {last_xcache!r}; "
                f"expected 'Hit from cloudfront' within the 90-second poll window."
            ),
        )
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Two HTTPS GETs against the same URL. The `X-Cache` header on the response tells you whether CloudFront went to S3 or served from the edge cache.

</details>

<details><summary>Hint 2 (direction)</summary>

`urlopen(Request(url, method="GET"))` returns a response with `.headers`, `.status`, and `.read()`. The `X-Cache` header is `Miss from cloudfront` on the first hit and `Hit from cloudfront` on the second. Sleep a few seconds between requests so CloudFront has time to populate the cache.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
print("First request:")
first_status, first_xcache, first_size = fetch_x_cache(url)
print(f"  Status={first_status}  X-Cache={first_xcache}  body={first_size}B")

time.sleep(5)

print("Second request:")
second_status, second_xcache, second_size = fetch_x_cache(url)
print(f"  Status={second_status}  X-Cache={second_xcache}  body={second_size}B")

assert "Hit from cloudfront" in second_xcache, f"Second X-Cache: {second_xcache}"
```

</details>

**Wallet check.** Still $0.00. Two GETs against CloudFront fit comfortably in the 10M-requests-per-month free-tier allowance.

## Cleanup

This is the slow one. The cloudfront_distribution adapter:

1. Reads the current ETag.
2. Updates the distribution config with `Enabled=False`.
3. Polls until `Status=Deployed` AND `Enabled=False` (10-15 minutes typical).
4. Re-reads the ETag (it changes when the disable lands).
5. Calls `delete_distribution` with the fresh ETag.

Then the OAC delete and S3 bucket delete take seconds. **Do not close the notebook tab while cleanup runs.** Total cleanup wall-clock is 12-20 minutes.

In [ ]:
# NBVAL_SKIP
# Cleanup. The cloudfront_distribution adapter handles the disable-then-delete
# cycle automatically; expect 12-20 minutes wall-clock.

import sys

print("Cleanup is about to start. CloudFront distribution disable + delete is")
print("the longest wait in the SAA cert track (10-15 minutes disable + 1-2 minutes")
print("delete). Keep the notebook tab open until cleanup prints `Cleanup complete`.")
print()

_rebuild_manifest()
result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_types = {"cloudfront_distribution"}
critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type in critical_types)
critical_destroyed = sum(
    1 for r in CLEANUP_MANIFEST if r.type in critical_types and r.id not in failed_ids
)
standard_total = len(CLEANUP_MANIFEST) - critical_total
standard_destroyed = standard_total - sum(
    1 for rid in failed_ids for r in CLEANUP_MANIFEST
    if r.id == rid and r.type not in critical_types
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** CloudFront's free tier swallowed the two GETs, the seed object was a few hundred bytes in S3, and OACs are free. The only cost was your time waiting for the deploy and the disable-then-delete cycle.

## Reflection

These are not graded. They are for you.

1. Walk through what CloudFront does when an edge location receives a request for an object that is not in cache. List each step: the cache lookup, the origin fetch via the OAC-signed request, the response delivery, the cache write. Where does the `X-Cache` header value get decided in that flow?

2. Your team wants to add a cache invalidation pattern so editorial content updates show up within 60 seconds globally instead of waiting for the 24-hour default TTL. Walk through the trade-offs of three options: (a) shorter TTLs on the cache policy, (b) `CreateInvalidation` API calls after each content update, (c) cache-busting query strings on the URL. When is each the right call?

## Exam-Style Practice

**Question 1 (Domain 3, CloudFront-to-S3 access pattern):**

You are putting CloudFront in front of a private S3 bucket. The bucket policy must allow CloudFront to fetch objects, but no other principal (not even other AWS accounts) should be able to read the bucket. Which mechanism is the AWS-recommended choice in 2026?

A. Origin Access Identity (OAI) with a bucket policy granting the OAI's IAM identity `s3:GetObject`.

B. Origin Access Control (OAC) with a bucket policy granting `Principal.Service=cloudfront.amazonaws.com` and a `Condition: StringEquals: AWS:SourceArn=<distribution ARN>`.

C. A signed URL generated by CloudFront for every request, with the bucket left public for the OAI to access.

D. A bucket policy granting `Principal.AWS=*` (public read) and relying on CloudFront WAF rules to filter requests.

<details><summary>Show answer</summary>

**Correct: B.**

- A is the legacy pattern. OAI is still supported but AWS recommends migrating to OAC, which supports SSE-KMS, HTTP methods beyond GET/HEAD, and SigV4 signed requests that OAI does not.
- B is correct: OAC with the `AWS:SourceArn` condition is the AWS-recommended pattern for private-S3-behind-CloudFront. The condition prevents the confused-deputy attack where another AWS account's distribution could fetch from your bucket.
- C is wrong: signed URLs control end-user access to CloudFront, not how CloudFront fetches from the origin. Leaving the bucket public is a security incident regardless of CloudFront in front of it.
- D is wrong: making the bucket public and filtering at WAF is fragile, expensive, and contradicts AWS Block Public Access. Use the bucket policy and OAC.

</details>

**Question 2 (Domain 3, CloudFront vs Global Accelerator vs Route 53 latency routing):**

A global SaaS app wants to reduce latency for users worldwide. The workload is a stateful gRPC API (not HTTP/HTTPS for static content), it uses two regional endpoints (us-east-1 and ap-southeast-2), and the team wants AWS-managed health checking with sub-30-second failover. Which AWS service is the best fit?

A. CloudFront with the two regional endpoints as origins and cache behaviors per path.

B. AWS Global Accelerator with the two regional endpoints as endpoint groups.

C. Route 53 with latency-based routing pointing at the two regional ALBs.

D. Direct Connect from each user's location to the nearest AWS Region.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: CloudFront is optimized for HTTP/HTTPS content delivery with edge caching. Stateful gRPC is not a CloudFront use case; gRPC is HTTP/2 but the workload here is API traffic, not cacheable content.
- B is correct: Global Accelerator provides static anycast IPs in front of regional endpoints, uses the AWS global backbone to route traffic to the nearest healthy endpoint, and ships with sub-30-second health-check-based failover. It is the right pick for stateful or non-HTTP workloads that need global routing.
- C is partially right but Route 53 health checks have a 30-second-to-3-minute failover window depending on configuration, and Route 53 is DNS-based so clients with stale DNS caches keep hitting the failed endpoint until their TTL expires.
- D is wrong: Direct Connect is a dedicated private network link from a customer location to AWS, not a global routing solution. End users worldwide do not have Direct Connect to AWS.

</details>